# Mask R-CNN Training for Aerial View Car Instance Segmentation

This notebook:
- decodes segmentation masks stored inside the JSON annotations
- builds a PyTorch dataset for instance segmentation
- trains a Mask R-CNN model on the aerial view car dataset
- saves checkpoints and loss curves
- runs a quick qualitative prediction on a sample image
    

## 1. Imports and Global Configuration
I have traded precision for speed here with torch.set_float32_matmul_precision('high') since I am running this code on RTX 3050 for faster training and inference.
Note that the precision loss will be very minimal. 

In [ ]:
import base64
import io
import json
import math
import os
import random
from glob import glob
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
import torchvision.transforms.v2 as T
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from tqdm.auto import tqdm

try:
    torch.set_float32_matmul_precision('high')
except AttributeError:
    pass

CLASS_MAP = {'car': 1}
CLASS_NAMES = ['__background__', 'car']
SEED = 42
MODEL_NAME = 'aerial_car_model'
NUM_CLASSES = len(CLASS_NAMES)
NUM_EPOCHS = 10
BATCH_SIZE = 2
LEARNING_RATE = 1e-4 
MOMENTUM = 0.9 
WEIGHT_DECAY = 5e-4 
TEST_SIZE = 0.2
SCORE_THRESHOLD = 0.5
CHECKPOINT_EVERY = 5 # Save checkpoint every N epochs

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'scripts':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_DIR = PROJECT_ROOT / 'dataset'
IMAGE_DIR = DATASET_DIR / 'images'
ANNOTATION_DIR = DATASET_DIR / 'annotations'
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints'
MODEL_DIR = PROJECT_ROOT / 'models'
LOSS_DIR = PROJECT_ROOT / 'losses_curves'

for output_dir in (CHECKPOINT_DIR, MODEL_DIR, LOSS_DIR):
    output_dir.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_workers = min(4, os.cpu_count() or 0) # Use up to 4 workers or the number of CPU cores available
use_pin_memory = device.type == 'cuda' # Pin memory for faster data transfer to GPU

print(f'Project root: {PROJECT_ROOT}')
print(f'Device: {device}')
print(f'DataLoader workers: {num_workers}')
    

## Helper tools 

The annotation files store encoded mask inside each JSON shape.
```
"shapes": [
    {
      "label": "car",
      "points": [
        [
          239.0,
          234.0
        ],
        [
          275.0,
          273.0
        ]
      ],
      "group_id": null,
      "description": "{\"score\": 0.9522280097007751, \"text\": \"car\"}",
      "shape_type": "mask",
      "flags": null,
      "mask": "iVBORw0KGgo..." # encoded mask
    } ],
    
```
These helpers decode the mask, resize it to the annotated bounding box, and place it back into the full image canvas.
    

In [ ]:
def decode_mask(mask_base64: str) -> np.ndarray:
    """Decode a base64-encoded PNG mask into a binary NumPy array."""
    mask_bytes = base64.b64decode(mask_base64)
    mask_image = Image.open(io.BytesIO(mask_bytes))
    mask = np.array(mask_image)
    return (mask > 0).astype(np.uint8)


def build_full_mask(mask_small: np.ndarray, box, image_height: int, image_width: int) -> np.ndarray:
    """Resize the cropped mask to its bounding box and paste it into a full-size mask."""
    x1, y1, x2, y2 = box
    x1 = int(np.floor(x1))
    y1 = int(np.floor(y1))
    x2 = int(np.ceil(x2))
    y2 = int(np.ceil(y2))

    target_width = max(x2 - x1, 1)
    target_height = max(y2 - y1, 1)

    resized_mask = cv2.resize(
        mask_small.astype(np.uint8),
        (target_width, target_height),
        interpolation=cv2.INTER_NEAREST,
    )

    full_mask = np.zeros((image_height, image_width), dtype=np.uint8)

    paste_x1 = max(x1, 0)
    paste_y1 = max(y1, 0)
    paste_x2 = min(x1 + resized_mask.shape[1], image_width)
    paste_y2 = min(y1 + resized_mask.shape[0], image_height)

    full_mask[paste_y1:paste_y2, paste_x1:paste_x2] = resized_mask[: paste_y2 - paste_y1, : paste_x2 - paste_x1]
    return full_mask


def parse_annotation(json_path: Path, image_height: int, image_width: int):
    """Parse one annotation file into boxes, labels, and binary masks."""
    with open(json_path, encoding='utf-8') as file:
        data = json.load(file)

    boxes = []
    labels = []
    masks = []

    for shape in data.get('shapes', []):
        label = shape['label']
        if label not in CLASS_MAP:
            continue

        (x1, y1), (x2, y2) = shape['points']
        box = [x1, y1, x2, y2]
        mask_small = decode_mask(shape['mask'])
        mask_full = build_full_mask(mask_small, box, image_height, image_width)

        boxes.append(box)
        labels.append(CLASS_MAP[label])
        masks.append(mask_full)

    return boxes, labels, masks
    

## Custom Dataset Class

This dataset reads each image and its matching annotation file, converts the annotations into the format expected by TorchVision detection models, and applies image transforms.
    

In [ ]:
class CarDataset(Dataset):
    def __init__(self, samples, transforms=None):
        self.samples = list(samples)
        self.transforms = transforms

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, annotation_path = self.samples[index]

        image = Image.open(image_path).convert('RGB')
        image_np = np.array(image)
        height, width = image_np.shape[:2]

        boxes, labels, masks = parse_annotation(annotation_path, height, width)
        if not boxes:
            raise ValueError(f'No valid annotations found for {annotation_path}')

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        masks = torch.as_tensor(np.stack(masks), dtype=torch.uint8)
        area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])

        target = {
            'boxes': boxes,
            'labels': labels,
            'masks': masks,
            'image_id': torch.tensor([index]),
            'area': area,
            'iscrowd': torch.zeros((len(labels),), dtype=torch.int64),
        }

        if self.transforms is not None:
            image = self.transforms(image)

        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))
    

This cell pairs images with JSON annotations, performs a reproducible train/validation split, and builds the PyTorch data loaders used during training.    

In [ ]:
def collect_samples(image_dir: Path, annotation_dir: Path):
    image_paths = {Path(path).stem: Path(path) for path in glob(str(image_dir / '*.jpg'))}
    annotation_paths = {Path(path).stem: Path(path) for path in glob(str(annotation_dir / '*.json'))}
    common_ids = sorted(image_paths.keys() & annotation_paths.keys())

    if not common_ids:
        raise FileNotFoundError(f'No matching image/annotation pairs found in {image_dir} and {annotation_dir}.')

    return [(image_paths[sample_id], annotation_paths[sample_id]) for sample_id in common_ids]


samples = collect_samples(IMAGE_DIR, ANNOTATION_DIR)
train_samples, val_samples = train_test_split(samples, test_size=TEST_SIZE, random_state=SEED)

image_transforms = T.Compose([
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
])

train_dataset = CarDataset(train_samples, transforms=image_transforms)
val_dataset = CarDataset(val_samples, transforms=image_transforms)

loader_kwargs = {
    'batch_size': BATCH_SIZE,
    'collate_fn': collate_fn,
    'num_workers': num_workers,
    'pin_memory': use_pin_memory,
}
if num_workers > 0:
    loader_kwargs['persistent_workers'] = True

train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)

print(f'Total samples: {len(samples)}')
print(f'Training samples: {len(train_dataset)}')
print(f'Validation samples: {len(val_dataset)}')
    

Let's confirm that the dataset, masks, and boxes were reconstructed correctly.

In [ ]:
def visualize_dataset_sample(dataset, sample_index=0):
    image, target = dataset[sample_index]
    image_np = image.permute(1, 2, 0).cpu().numpy()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(image_np)
    axes[0].set_title('Input image')
    axes[0].axis('off')

    combined_mask = target['masks'].sum(dim=0).clamp(max=1).cpu().numpy()
    axes[1].imshow(combined_mask, cmap='gray')
    axes[1].set_title(f"Combined masks ({len(target['boxes'])} instances)")
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()


visualize_dataset_sample(train_dataset, sample_index=0)
    

Looks like images and masks were constructed properly so, we start from the pretrained `maskrcnn_resnet50_fpn_v2` backbone and replace the classification and mask heads so the model predicts our dataset classes.

In [ ]:
def build_model(num_classes: int) -> torchvision.models.detection.MaskRCNN:
    model = torchvision.models.detection.maskrcnn_resnet50_fpn_v2(weights='DEFAULT')

    box_in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(box_in_features, num_classes)

    mask_in_channels = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(mask_in_channels, hidden_layer, num_classes)

    return model


model = build_model(NUM_CLASSES).to(device)
trainable_params = [parameter for parameter in model.parameters() if parameter.requires_grad]

optimizer = torch.optim.SGD(
    trainable_params,
    lr=LEARNING_RATE,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)

lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=1.0)

print(model.__class__.__name__)
print(f'Trainable parameters: {sum(parameter.numel() for parameter in trainable_params):,}')
    

## Training and Validation Utilities

These helper functions keep the training loop compact.
They handle device transfer, warm-up scheduling, loss aggregation, plotting, and checkpoint saving.
    

In [ ]:
def move_batch_to_device(images, targets, device):
    images = [image.to(device) for image in images]
    targets = [{key: value.to(device) for key, value in target.items()} for target in targets]
    return images, targets


TRAIN_KEYS = [
    'loss_classifier',
    'loss_box_reg',
    'loss_mask',
    'loss_objectness',
    'loss_rpn_box_reg',
    'loss_sum',
]
VAL_KEYS = [f'val_{key}' for key in TRAIN_KEYS]


def run_training_epoch(model, optimizer, data_loader, device, epoch: int):
    model.train()
    history = {key: [] for key in TRAIN_KEYS}

    warmup_scheduler = None
    if epoch == 0:
        warmup_iters = max(min(1000, len(data_loader) - 1), 1)
        warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
            optimizer,
            start_factor=1.0 / 1000,
            total_iters=warmup_iters,
        )

    for images, targets in tqdm(data_loader, desc=f'Train {epoch + 1}', leave=False):
        images, targets = move_batch_to_device(images, targets, device)
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        loss_value = losses.item()

        if not math.isfinite(loss_value):
            raise RuntimeError(f'Non-finite loss encountered: {loss_value}. Loss dict: {loss_dict}')

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        if warmup_scheduler is not None:
            warmup_scheduler.step()

        for key, value in loss_dict.items():
            history[key].append(value.item())
        history['loss_sum'].append(loss_value)

    return history


def run_validation_epoch(model, data_loader, device, epoch: int):
    history = {key: [] for key in VAL_KEYS}

    with torch.no_grad():
        model.train()
        for images, targets in tqdm(data_loader, desc=f'Val {epoch + 1}', leave=False):
            images, targets = move_batch_to_device(images, targets, device)
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            for key, value in loss_dict.items():
                history[f'val_{key}'].append(value.item())
            history['val_loss_sum'].append(losses.item())

    return history


def average_history(history):
    return {key: float(np.mean(values)) for key, values in history.items()}


def initialize_loss_store():
    return {key: [] for key in TRAIN_KEYS + VAL_KEYS}


def update_loss_store(loss_store, train_epoch_history, val_epoch_history):
    train_summary = average_history(train_epoch_history)
    val_summary = average_history(val_epoch_history)
    for key, value in {**train_summary, **val_summary}.items():
        loss_store[key].append(value)
    return train_summary, val_summary


def plot_loss_curves(loss_store, save_path: Path):
    fig, axes = plt.subplots(3, 2, figsize=(12, 12))
    axes = axes.flatten()

    metric_pairs = [
        ('loss_classifier', 'val_loss_classifier'),
        ('loss_box_reg', 'val_loss_box_reg'),
        ('loss_mask', 'val_loss_mask'),
        ('loss_objectness', 'val_loss_objectness'),
        ('loss_rpn_box_reg', 'val_loss_rpn_box_reg'),
        ('loss_sum', 'val_loss_sum'),
    ]

    for axis, (train_key, val_key) in zip(axes, metric_pairs):
        axis.plot(loss_store[train_key], label=train_key, color='tab:blue')
        axis.plot(loss_store[val_key], label=val_key, color='tab:red')
        axis.set_title(train_key.replace('loss_', '').replace('_', ' ').title())
        axis.set_xlabel('Epoch')
        axis.set_ylabel('Loss')
        axis.legend(fontsize='small')
        axis.grid(alpha=0.3)

    fig.suptitle('Mask R-CNN training history', fontsize=16)
    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


def save_checkpoint(path: Path, epoch: int, model, optimizer, loss_store):
    torch.save(
        {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss_store,
        },
        path,
    )
    

## Train the Model

This is the main training loop.
For each epoch, we train, validate, update the loss history, save the best checkpoint, periodically save a rolling checkpoint, and write the final model to disk.
    

In [ ]:
loss_store = initialize_loss_store()
best_checkpoint_path = CHECKPOINT_DIR / f'{MODEL_NAME}_checkpoint_min.pt'
latest_checkpoint_path = CHECKPOINT_DIR / f'{MODEL_NAME}_checkpoint.pt'
final_model_path = MODEL_DIR / f'{MODEL_NAME}.pt'
loss_curve_path = LOSS_DIR / f'{MODEL_NAME}_losses_curve.png'

best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    current_lr = lr_scheduler.get_last_lr()[0]
    print(f'Epoch {epoch + 1}/{NUM_EPOCHS} | lr={current_lr:.6f}')

    train_epoch_history = run_training_epoch(model, optimizer, train_loader, device, epoch)
    val_epoch_history = run_validation_epoch(model, val_loader, device, epoch)
    train_summary, val_summary = update_loss_store(loss_store, train_epoch_history, val_epoch_history)

    lr_scheduler.step()

    plot_loss_curves(loss_store, loss_curve_path)

    print(f"Train total loss: {train_summary['loss_sum']:.4f}")
    print(f"Val total loss:   {val_summary['val_loss_sum']:.4f}")

    if val_summary['val_loss_sum'] < best_val_loss:
        save_checkpoint(best_checkpoint_path, epoch, model, optimizer, loss_store)
        best_val_loss = val_summary['val_loss_sum']

    if (epoch + 1) % CHECKPOINT_EVERY == 0:
        save_checkpoint(latest_checkpoint_path, epoch, model, optimizer, loss_store)

torch.save(model, final_model_path)

print(f'Best checkpoint: {best_checkpoint_path}')
print(f'Latest checkpoint: {latest_checkpoint_path}')
print(f'Final model: {final_model_path}')
print(f'Loss curve: {loss_curve_path}')
    

## Inference Helpers

After training, we can run the model on a single image and overlay the predicted masks and class names.
This gives a quick qualitative check of how well the model learned.
    

In [ ]:
def generate_colors():
    colors = []
    for r in range(0, 255, 120):
        for g in range(0, 255, 120):
            for b in range(0, 255, 120):
                if (r, g, b) != (0, 0, 0):
                    colors.append((r, g, b))
    random.shuffle(colors)
    return colors


def colorize_mask(mask: np.ndarray, color):
    color_mask = cv2.cvtColor(mask.astype(np.uint8), cv2.COLOR_GRAY2BGR)
    color_mask[mask > 0] = color
    return color_mask


def predict_instances(image_bgr: np.ndarray, model, class_names, threshold=SCORE_THRESHOLD):
    rgb_image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    image_tensor = torch.from_numpy(rgb_image).permute(2, 0, 1).float().div(255.0).to(device)

    model.eval()
    with torch.no_grad():
        prediction = model([image_tensor])[0]

    scores = prediction['scores'].detach().cpu().numpy()
    valid_count = int((scores > threshold).sum())

    if valid_count == 0:
        return [], [], []

    masks = (prediction['masks'][:valid_count, 0] > 0.5).cpu().numpy().astype(np.uint8)
    boxes = prediction['boxes'][:valid_count].detach().cpu().numpy().astype(int)
    labels = prediction['labels'][:valid_count].detach().cpu().numpy()
    class_labels = [class_names[label] for label in labels]

    return masks, boxes, class_labels


def render_predictions(image_bgr: np.ndarray, masks, boxes, classes):
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    colors = generate_colors()

    for index, mask in enumerate(masks):
        color = colors[index % len(colors)]
        image_rgb = cv2.addWeighted(image_rgb, 1.0, colorize_mask(mask, color), 0.5, 0)

        x1, y1, x2, y2 = boxes[index]
        cv2.rectangle(image_rgb, (x1, y1), (x2, y2), color, 2)
        cv2.putText(image_rgb, classes[index], (x1, max(y1 - 10, 0)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    return image_rgb


def plot_prediction(image_path: Path, model, class_names, threshold=SCORE_THRESHOLD):
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        raise FileNotFoundError(f'Could not read image: {image_path}')

    masks, boxes, classes = predict_instances(image_bgr, model, class_names, threshold=threshold)
    rendered = render_predictions(image_bgr, masks, boxes, classes)

    plt.figure(figsize=(10, 6))
    plt.imshow(rendered)
    plt.title(f'Prediction for {image_path.name}')
    plt.axis('off')
    plt.show()
    

## Visualize a Sample Prediction 

In [ ]:
sample_image_path = train_samples[0][0]
plot_prediction(sample_image_path, model, CLASS_NAMES, threshold=SCORE_THRESHOLD)
    